# McCulloch-Pitts Neuron: Implementation

A Python implementation of the McCulloch-Pitts (MCP) neuron model from the 1943 paper
*"A Logical Calculus of the Ideas Immanent in Nervous Activity"*.

We will:
1. Implement the basic MCP neuron
2. Build logic gates (AND, OR, NOT, NAND, NOR, XOR)
3. Reproduce Figure 1 from the paper
4. Simulate a temporal network (heat/cold sensation example)
5. Demonstrate computational universality

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product
from typing import Optional

## 1. The McCulloch-Pitts Neuron

The MCP neuron is defined by the paper's 5 assumptions:

- **Inputs**: Binary values (0 or 1)
- **Output**: Binary (0 or 1) — "all-or-none"
- **Threshold** $\theta$: The neuron fires (output=1) if the sum of excitatory inputs $\geq \theta$
- **Inhibition**: If ANY inhibitory input is active, the neuron CANNOT fire (absolute inhibition)
- **Time**: The neuron operates in discrete time steps; output at time $t$ depends on inputs at time $t-1$

$$
y(t) = \begin{cases}
0 & \text{if any inhibitory input is 1} \\
1 & \text{if } \sum x_i^{\text{exc}} \geq \theta \text{ and no inhibitory input is 1} \\
0 & \text{otherwise}
\end{cases}
$$

In [ ]:
class MCPNeuron:
    """McCulloch-Pitts Neuron Model.

    A binary threshold neuron with excitatory and inhibitory inputs.
    Implements the model described in McCulloch & Pitts (1943).

    Parameters
    ----------
    threshold : int
        Minimum number of active excitatory inputs needed to fire.
    n_excitatory : int
        Number of excitatory input connections.
    n_inhibitory : int
        Number of inhibitory input connections.
    name : str, optional
        Label for this neuron.
    """

    def __init__(
        self,
        threshold: int,
        n_excitatory: int,
        n_inhibitory: int = 0,
        name: Optional[str] = None,
    ):
        self.threshold = threshold
        self.n_excitatory = n_excitatory
        self.n_inhibitory = n_inhibitory
        self.name = name or f"MCP(θ={threshold})"

    def activate(self, excitatory: list[int], inhibitory: Optional[list[int]] = None) -> int:
        """Compute the neuron's output for given inputs.

        Parameters
        ----------
        excitatory : list of int
            Excitatory inputs (each 0 or 1).
        inhibitory : list of int, optional
            Inhibitory inputs (each 0 or 1). If any is 1, output is 0.

        Returns
        -------
        int
            1 if neuron fires, 0 otherwise.
        """
        if inhibitory is None:
            inhibitory = []

        # Absolute inhibition: if ANY inhibitory input is active, neuron cannot fire
        if any(i == 1 for i in inhibitory):
            return 0

        # Sum excitatory inputs and compare to threshold
        if sum(excitatory) >= self.threshold:
            return 1

        return 0

    def truth_table(self) -> None:
        """Print the complete truth table for this neuron."""
        exc_combos = list(product([0, 1], repeat=self.n_excitatory))
        inh_combos = list(product([0, 1], repeat=self.n_inhibitory)) if self.n_inhibitory > 0 else [()]

        # Header
        exc_labels = [f"e{i+1}" for i in range(self.n_excitatory)]
        inh_labels = [f"i{i+1}" for i in range(self.n_inhibitory)]
        header = "  ".join(exc_labels + inh_labels + ["| OUT"])
        print(f"\n{self.name} (θ={self.threshold})")
        print(header)
        print("-" * len(header))

        for exc in exc_combos:
            for inh in inh_combos:
                output = self.activate(list(exc), list(inh))
                values = list(exc) + list(inh) + [output]
                row = "   ".join(str(v) for v in values[:-1])
                print(f"{row}  |  {output}")


print("MCPNeuron class defined.")

## 2. Logic Gates with MCP Neurons

The key insight of the paper: simple MCP neurons can implement any Boolean logic gate.
Let's build each one and verify with truth tables.

### 2.1 AND Gate

Fires only when **all** inputs are active. Set threshold = number of inputs.

Corresponds to **Figure 1(c)** in the paper: $N_3(t) = N_1(t-1) \land N_2(t-1)$

In [ ]:
and_gate = MCPNeuron(threshold=2, n_excitatory=2, name="AND")
and_gate.truth_table()

### 2.2 OR Gate

Fires when **any** input is active. Set threshold = 1.

Corresponds to **Figure 1(b)**: $N_3(t) = N_1(t-1) \lor N_2(t-1)$

In [ ]:
or_gate = MCPNeuron(threshold=1, n_excitatory=2, name="OR")
or_gate.truth_table()

### 2.3 NOT Gate (Inverter)

The NOT gate uses **inhibition**. One inhibitory input; the neuron has a threshold of 0
(it always *wants* to fire) but is prevented by the inhibitory input.

Implementation: a neuron with threshold=1 that receives a constant excitatory input of 1
(a "tonic" input) and the signal to negate as an inhibitory input.

In [ ]:
class NOTGate:
    """NOT gate using an MCP neuron with inhibition.

    Uses a tonic (always-on) excitatory input and the signal
    to negate as an inhibitory input.
    """

    def __init__(self):
        self.neuron = MCPNeuron(threshold=1, n_excitatory=1, n_inhibitory=1, name="NOT")

    def activate(self, x: int) -> int:
        """Return the negation of input x."""
        return self.neuron.activate(excitatory=[1], inhibitory=[x])


not_gate = NOTGate()
print("NOT Gate")
print("x | OUT")
print("-------")
for x in [0, 1]:
    print(f"{x} |  {not_gate.activate(x)}")

### 2.4 NAND Gate

NOT-AND: fires unless all inputs are active.

This is important because NAND is **functionally complete** — any Boolean function
can be built from NAND gates alone.

Corresponds to **Figure 1(d)**: $N_3(t) = N_1(t-1) \land \lnot N_2(t-1)$
(though 1(d) is specifically AND-NOT, the principle is the same)

In [ ]:
class NANDGate:
    """NAND gate: fires unless ALL inputs are 1.

    Implementation: tonic input + inhibitory AND.
    When both inputs are 1, the AND neuron fires and inhibits the output neuron.
    """

    def __init__(self):
        self.and_neuron = MCPNeuron(threshold=2, n_excitatory=2, name="AND_internal")
        self.output_neuron = MCPNeuron(threshold=1, n_excitatory=1, n_inhibitory=1, name="NAND_out")

    def activate(self, x1: int, x2: int) -> int:
        and_result = self.and_neuron.activate([x1, x2])
        return self.output_neuron.activate(excitatory=[1], inhibitory=[and_result])


nand_gate = NANDGate()
print("NAND Gate")
print("x1  x2 | OUT")
print("------------")
for x1, x2 in product([0, 1], repeat=2):
    print(f" {x1}   {x2}  |  {nand_gate.activate(x1, x2)}")

### 2.5 XOR Gate — The Crucial Test

XOR (exclusive or) cannot be computed by a single MCP neuron. This foreshadows
Minsky & Papert's 1969 critique of the perceptron.

However, McCulloch & Pitts showed that a **network** of neurons can compute it.

XOR = (x1 AND NOT x2) OR (NOT x1 AND x2)

This requires a two-layer network — a precursor to the idea of "hidden layers".

In [ ]:
class XORGate:
    """XOR gate built from MCP neurons.

    XOR(x1, x2) = (x1 AND NOT x2) OR (NOT x1 AND x2)

    Architecture:
        Layer 1: Two AND-NOT neurons (with inhibitory inputs)
            n1: fires if x1=1 and x2=0  (x1 excitatory, x2 inhibitory, θ=1)
            n2: fires if x2=1 and x1=0  (x2 excitatory, x1 inhibitory, θ=1)
        Layer 2: One OR neuron
            output: fires if n1 or n2 fires (θ=1)
    """

    def __init__(self):
        # Layer 1: AND-NOT neurons
        self.n1 = MCPNeuron(threshold=1, n_excitatory=1, n_inhibitory=1, name="x1 AND NOT x2")
        self.n2 = MCPNeuron(threshold=1, n_excitatory=1, n_inhibitory=1, name="NOT x1 AND x2")
        # Layer 2: OR neuron
        self.output = MCPNeuron(threshold=1, n_excitatory=2, name="OR")

    def activate(self, x1: int, x2: int) -> int:
        # Layer 1
        h1 = self.n1.activate(excitatory=[x1], inhibitory=[x2])  # x1 AND NOT x2
        h2 = self.n2.activate(excitatory=[x2], inhibitory=[x1])  # x2 AND NOT x1
        # Layer 2
        return self.output.activate(excitatory=[h1, h2])


xor_gate = XORGate()
print("XOR Gate (two-layer network)")
print("x1  x2 | OUT")
print("------------")
for x1, x2 in product([0, 1], repeat=2):
    print(f" {x1}   {x2}  |  {xor_gate.activate(x1, x2)}")

## 3. Summary of All Gates

Let's visualize all gates side by side to see how threshold and inhibition
control the logical function.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("McCulloch-Pitts Logic Gates", fontsize=16, fontweight="bold")

gates = {
    "AND (θ=2)": lambda x1, x2: and_gate.activate([x1, x2]),
    "OR (θ=1)": lambda x1, x2: or_gate.activate([x1, x2]),
    "NOT": None,  # handled separately
    "NAND": lambda x1, x2: nand_gate.activate(x1, x2),
    "XOR (network)": lambda x1, x2: xor_gate.activate(x1, x2),
    "AND-NOT": lambda x1, x2: MCPNeuron(1, 1, 1).activate([x1], [x2]),
}

inputs = list(product([0, 1], repeat=2))

for ax, (name, gate_fn) in zip(axes.flat, gates.items()):
    if name == "NOT":
        # Special case: 1 input
        table_data = [[x, not_gate.activate(x)] for x in [0, 1]]
        colors = [["#ffffff", "#90EE90" if row[1] else "#FFB3B3"] for row in table_data]
        table = ax.table(
            cellText=table_data,
            colLabels=["x", "OUT"],
            cellColours=colors,
            loc="center",
            cellLoc="center",
        )
    else:
        results = [gate_fn(x1, x2) for x1, x2 in inputs]
        table_data = [[x1, x2, r] for (x1, x2), r in zip(inputs, results)]
        colors = [
            ["#ffffff", "#ffffff", "#90EE90" if r else "#FFB3B3"]
            for r in results
        ]
        table = ax.table(
            cellText=table_data,
            colLabels=["x1", "x2", "OUT"],
            cellColours=colors,
            loc="center",
            cellLoc="center",
        )

    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 1.8)
    ax.set_title(name, fontsize=13, fontweight="bold", pad=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig("logic_gates_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("All gates verified!")

## 4. Temporal Network: Heat/Cold Sensation

From the paper (p. 106): when a cold object is briefly held to skin, you feel **heat**;
if held longer, you feel only **cold**.

The paper models this with:
- $N_1$: heat receptor
- $N_2$: cold receptor
- $N_3$: neuron whose activity = sensation of heat
- $N_4$: neuron whose activity = sensation of cold

$$N_3(t) = N_1(t-1) \lor [N_2(t-3) \land \lnot N_2(t-2)]$$
$$N_4(t) = N_2(t-2) \land N_2(t-1)$$

In words:
- Heat sensation: either the heat receptor fired, OR cold started recently (transient)
- Cold sensation: cold receptor has been active for at least 2 consecutive steps

In [ ]:
def simulate_heat_cold(n_steps: int, cold_start: int, cold_end: int) -> dict:
    """Simulate the heat/cold sensation network from the paper.

    Parameters
    ----------
    n_steps : int
        Total number of time steps to simulate.
    cold_start : int
        Time step when cold stimulus begins.
    cold_end : int
        Time step when cold stimulus ends.

    Returns
    -------
    dict
        Time series for N1 (heat receptor), N2 (cold receptor),
        N3 (heat sensation), N4 (cold sensation).
    """
    N1 = np.zeros(n_steps, dtype=int)  # Heat receptor
    N2 = np.zeros(n_steps, dtype=int)  # Cold receptor
    N3 = np.zeros(n_steps, dtype=int)  # Heat sensation
    N4 = np.zeros(n_steps, dtype=int)  # Cold sensation

    # Cold stimulus active from cold_start to cold_end
    N2[cold_start:cold_end] = 1

    for t in range(3, n_steps):
        # N3(t) = N1(t-1) OR [N2(t-3) AND NOT N2(t-2)]
        # Heat sensation: heat receptor fired, or cold just started (transient)
        cold_onset = N2[t - 3] and not N2[t - 2]
        N3[t] = int(N1[t - 1] or cold_onset)

        # N4(t) = N2(t-2) AND N2(t-1)
        # Cold sensation: cold receptor active for 2+ steps
        N4[t] = int(N2[t - 2] and N2[t - 1])

    return {"N1_heat_receptor": N1, "N2_cold_receptor": N2,
            "N3_heat_sensation": N3, "N4_cold_sensation": N4}


# Scenario 1: Brief cold stimulus (2 steps)
brief = simulate_heat_cold(n_steps=15, cold_start=3, cold_end=5)

# Scenario 2: Prolonged cold stimulus (8 steps)
prolonged = simulate_heat_cold(n_steps=15, cold_start=3, cold_end=11)

print("Simulations complete.")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

scenarios = [
    ("Brief cold stimulus (2 steps)", brief),
    ("Prolonged cold stimulus (8 steps)", prolonged),
]

colors = {"N2_cold_receptor": "#4A90D9", "N3_heat_sensation": "#E74C3C", "N4_cold_sensation": "#3498DB"}
labels = {"N2_cold_receptor": "N₂: Cold receptor (input)",
          "N3_heat_sensation": "N₃: Heat sensation",
          "N4_cold_sensation": "N₄: Cold sensation"}

for ax, (title, data) in zip(axes, scenarios):
    t = np.arange(len(data["N2_cold_receptor"]))

    for key in ["N2_cold_receptor", "N3_heat_sensation", "N4_cold_sensation"]:
        offset = {"N2_cold_receptor": 4, "N3_heat_sensation": 2, "N4_cold_sensation": 0}[key]
        ax.fill_between(t, offset, offset + np.array(data[key]) * 0.8,
                        alpha=0.7, color=colors[key], step="mid", label=labels[key])
        ax.axhline(y=offset, color="gray", linewidth=0.3)

    ax.set_yticks([0.4, 2.4, 4.4])
    ax.set_yticklabels(["Cold\nsensation", "Heat\nsensation", "Cold\nreceptor"], fontsize=10)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlim(-0.5, 14.5)
    ax.set_xticks(range(15))
    ax.grid(axis="x", alpha=0.3)

axes[1].set_xlabel("Time step (t)", fontsize=12)
axes[0].legend(loc="upper right", fontsize=9)

plt.suptitle("McCulloch-Pitts: Heat/Cold Sensation Network", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("heat_cold_simulation.png", dpi=150, bbox_inches="tight")
plt.show()

**Observations:**

- **Brief cold stimulus**: The cold receptor activates briefly. Because cold starts and then the
  onset condition triggers, you get a brief heat sensation. Cold sensation requires 2 consecutive
  steps of cold, so it may appear briefly or not at all.
- **Prolonged cold stimulus**: The initial onset still triggers a brief heat sensation (the transient),
  but then sustained cold receptor activity produces a sustained cold sensation.

This matches the real phenomenon: a brief touch of cold feels like heat!

## 5. Computational Universality

McCulloch & Pitts proved that their neuron networks can compute anything a Turing machine can.
We demonstrate this practically by building a **half adder** — a circuit that adds two bits.

A half adder computes:
- **Sum** = XOR(A, B)
- **Carry** = AND(A, B)

In [ ]:
class HalfAdder:
    """Half adder built entirely from MCP neurons.

    Adds two single-bit numbers A and B, producing Sum and Carry.
    """

    def __init__(self):
        self.xor = XORGate()  # For Sum
        self.and_gate = MCPNeuron(threshold=2, n_excitatory=2, name="Carry")  # For Carry

    def add(self, a: int, b: int) -> tuple[int, int]:
        """Add two bits.

        Returns
        -------
        tuple of (sum_bit, carry_bit)
        """
        s = self.xor.activate(a, b)
        c = self.and_gate.activate([a, b])
        return s, c


adder = HalfAdder()
print("Half Adder (built from MCP neurons)")
print("  A   B  | Sum  Carry  (decimal)")
print("-" * 35)
for a, b in product([0, 1], repeat=2):
    s, c = adder.add(a, b)
    decimal = c * 2 + s
    print(f"  {a}   {b}  |  {s}    {c}      ({a}+{b}={decimal})")

## 6. Limitations and Path Forward

### What the MCP neuron **cannot** do:
1. **Learn** — weights and thresholds are fixed by design (Assumption 5)
2. **Handle continuous values** — inputs and outputs are strictly binary
3. **Adapt** — no mechanism for changing behavior based on experience

### What comes next in the reading list:
- **Hebb (1949)**: Proposed that synaptic connections strengthen when neurons fire together
  ("cells that fire together wire together") — the first learning rule
- **Rosenblatt (1958)**: The Perceptron — added **learnable weights** to the MCP neuron,
  creating the first trainable neural network
- **Minsky & Papert (1969)**: Proved limitations of single-layer perceptrons
  (e.g., can't learn XOR without hidden layers)

## Key Takeaways from Implementation

1. The MCP neuron is remarkably simple: sum inputs, compare to threshold, check inhibition
2. Despite this simplicity, networks of MCP neurons can compute **any logical function**
3. **Inhibition is essential** — without it, NOT cannot be computed, and universality breaks
4. **Multi-layer networks** are needed for functions like XOR — a single neuron isn't enough
5. **Temporal behavior** (memory, sequences) emerges naturally from network structure
6. The model has no learning — every connection must be designed by hand